# Issue #168 and #209: setting atom charges

**Goal**: to modify ``__init__`` and ``charge`` methods in ``Atom`` class (*structural_units.py*) to allow for setting of charge during atom initialization, and also independent of whether a Coulombic interaction already exists.

In [1]:
# Import modules
from MDMC.MD.simulation import Universe
from MDMC.MD.structural_units import Atom, Coulombic
from MDMC.MD.interaction_functions import Coulomb

# Define the universe
universe = Universe(10.0)

In [2]:
# Initialize the atoms with the charge; a warning is given to the user
H1 = Atom('H', charge=0.8)
H2 = H1.copy(position=[1., 1., 1.])
O1 = Atom('O', position=[2., 2., 2.], charge=0.6)
O2 = O1.copy(position=[3., 3., 3.])

# Add the atoms to the universe
[universe.add_structural_unit(i) for i in (H1, H2, O1, O2)]
universe.structure_list

MDMC/MD/structural_units.py:715: UserWarning: WARNING: Coulombic interaction for the Atom object initialized with the set charge value.
  warnings.warn('WARNING: Coulombic interaction for the Atom '


[H atom,  ID: 1  charge: 0.8 e,  interactions: ['Coulombic'],
 O atom,  ID: 4  charge: 0.6 e,  interactions: ['Coulombic'],
 H atom,  ID: 2  charge: 0.8 e,  interactions: ['Coulombic'],
 O atom,  ID: 3  charge: 0.6 e,  interactions: ['Coulombic']]

In [3]:
# Charges can also be set so that charges of all atoms of that type change,
# provided they are a copy
O1.charge = -101010
H2.charge = 77777
universe.structure_list

[H atom,  ID: 1  charge: 77777.0 e,  interactions: ['Coulombic'],
 O atom,  ID: 4  charge: -101010.0 e,  interactions: ['Coulombic'],
 H atom,  ID: 2  charge: 77777.0 e,  interactions: ['Coulombic'],
 O atom,  ID: 3  charge: -101010.0 e,  interactions: ['Coulombic']]

In [4]:
# Adding a new atom of a different type
C1 = Atom('C')

# Setting its charge outside of it's initialization
C1.charge = 666

# Adding it to the universe
universe.add_structural_unit(C1)
universe.structure_list

[H atom,  ID: 1  charge: 77777.0 e,  interactions: ['Coulombic'],
 O atom,  ID: 4  charge: -101010.0 e,  interactions: ['Coulombic'],
 H atom,  ID: 2  charge: 77777.0 e,  interactions: ['Coulombic'],
 C atom,  ID: 5  charge: 666.0 e,  interactions: ['Coulombic'],
 O atom,  ID: 3  charge: -101010.0 e,  interactions: ['Coulombic']]

### .... here setting the charge outside atom initialization didn't show the warning as it did above (need to change this).

In [5]:
# When adding a copy atom, the charge cannot yet be edited.
C2 = C1.copy(position=[4., 4., 4.,], charge=5678)
universe.add_structural_unit(C2)
universe.structure_list

TypeError: copy() got an unexpected keyword argument 'charge'

### .... should the *copy()* method in the *StructuralUnit* class be edited so that the charges can be changed upon copying an atom?

In [6]:
# Adding a new Carbon atom with a different charge
C3 = Atom('C', charge=-12345, atom_type=C1.atom_type)
universe.add_structural_unit(C3)
universe.structure_list

[H atom,  ID: 1  charge: 77777.0 e,  interactions: ['Coulombic'],
 O atom,  ID: 3  charge: -101010.0 e,  interactions: ['Coulombic'],
 O atom,  ID: 4  charge: -101010.0 e,  interactions: ['Coulombic'],
 C atom,  ID: 6  charge: -12345.0 e,  interactions: ['Coulombic'],
 H atom,  ID: 2  charge: 77777.0 e,  interactions: ['Coulombic'],
 C atom,  ID: 5  charge: 666.0 e,  interactions: ['Coulombic']]

In [7]:
# C1 and C3 are the same type
C1.atom_type == C3.atom_type

True

In [8]:
# Modifying the charge of C3 doesn't modify charge of C1 as C3 isn't a copy,
# despite them both being of the same type.
C3.charge = 0
universe.structure_list

[H atom,  ID: 1  charge: 77777.0 e,  interactions: ['Coulombic'],
 O atom,  ID: 3  charge: -101010.0 e,  interactions: ['Coulombic'],
 O atom,  ID: 4  charge: -101010.0 e,  interactions: ['Coulombic'],
 C atom,  ID: 6  charge: 0.0 e,  interactions: ['Coulombic'],
 H atom,  ID: 2  charge: 77777.0 e,  interactions: ['Coulombic'],
 C atom,  ID: 5  charge: 666.0 e,  interactions: ['Coulombic']]

### .... do we want to implement a mechanism by which charges for all atoms of the same type change upon modification of one of them, or is relying on them being copies of eachother sufficient?

## Side note

The examples given within the docstring of the *Coulombic* class in *structural_units.py* aren't up to date:

In [9]:
from MDMC.MD.simulation import Universe
from MDMC.MD.structural_units import Atom

O1 = Atom('O', atom_type=1)
universe = Universe(10.0)

# Creating the interaction; all 3 of these methods work fine:
c_O = Coulombic(atoms=O1, function=Coulomb((0.5, 'e')))
c_O = Coulombic(atoms=[O1], atom_types=[O1.atom_type], function=Coulomb((-0.8, 'e')))
c_O = Coulombic(universe, O1.atom_type, charge=0.42)

# But the examples given in the Coulombic class docstring don't:
O = Atom('O', atom_type=1)
O_coulombic = Coulombic(O.atom_type, charge=-0.84)
O_coulombic = Coulombic(O.atom_type, function=Coulomb(-0.84))

TypeError: Coulombic takes either atom_types or atoms as parameters